In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

MODEL_PATH = r"C:\Users\Admin\Desktop\CropGuard\CropGuard\ml\models\cropguard_seg_aug.pt"
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)

class_names = checkpoint['class_names']
num_classes = len(class_names)
print(f"Loaded model with {num_classes} classes, val_acc at save time: {checkpoint['val_acc']:.2f}%")

# Rebuild the model architecture (must match training exactly)
model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

# Validate the PlantDoc -> PlantVillage mapping against actual class names
PLANTDOC_TO_PLANTVILLAGE = {
    "Apple_leaf": "Apple___healthy",
    "Apple_rust_leaf": "Apple___Cedar_apple_rust",
    "Apple_Scab_Leaf": "Apple___Apple_scab",
    "Bell_pepper_leaf": "Pepper,_bell___healthy",
    "Bell_pepper_leaf_spot": "Pepper,_bell___Bacterial_spot",
    "Blueberry_leaf": "Blueberry___healthy",
    "Cherry_leaf": "Cherry_(including_sour)___healthy",
    "Corn_Gray_leaf_spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_leaf_blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_rust_leaf": "Corn_(maize)___Common_rust_",
    "grape_leaf": "Grape___healthy",
    "grape_leaf_black_rot": "Grape___Black_rot",
    "Peach_leaf": "Peach___healthy",
    "Potato_leaf_early_blight": "Potato___Early_blight",
    "Potato_leaf_late_blight": "Potato___Late_blight",
    "Raspberry_leaf": "Raspberry___healthy",
    "Soyabean_leaf": "Soybean___healthy",
    "Squash_Powdery_mildew_leaf": "Squash___Powdery_mildew",
    "Strawberry_leaf": "Strawberry___healthy",
    "Tomato_Early_blight_leaf": "Tomato___Early_blight",
    "Tomato_leaf": "Tomato___healthy",
    "Tomato_leaf_bacterial_spot": "Tomato___Bacterial_spot",
    "Tomato_leaf_late_blight": "Tomato___Late_blight",
    "Tomato_leaf_mosaic_virus": "Tomato___Tomato_mosaic_virus",
    "Tomato_leaf_yellow_virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato_mold_leaf": "Tomato___Leaf_Mold",
    "Tomato_Septoria_leaf_spot": "Tomato___Septoria_leaf_spot",
    "Tomato_two_spotted_spider_mites_leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
}

missing = [pv_name for pv_name in PLANTDOC_TO_PLANTVILLAGE.values() if pv_name not in class_names]
if missing:
    print(f"WARNING: {len(missing)} mapped names NOT found in class_names:")
    for m in missing:
        print(f"  - {m}")
else:
    print("All mapped PlantVillage class names validated successfully.")

Using device: cuda
Loaded model with 38 classes, val_acc at save time: 93.87%
All mapped PlantVillage class names validated successfully.


In [2]:
PLANTDOC_DIR = r"C:\Users\Admin\Desktop\CropGuard\CropGuard\ml\data\plantdoc_segmented\test"

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

plantdoc_dataset = datasets.ImageFolder(PLANTDOC_DIR, transform=eval_transform)
plantdoc_loader = DataLoader(plantdoc_dataset, batch_size=32, shuffle=False, num_workers=4)

print(f"PlantDoc test set: {len(plantdoc_dataset)} images, {len(plantdoc_dataset.classes)} classes")
print(plantdoc_dataset.classes)

# Build index mapping: PlantDoc class idx -> PlantVillage class idx
plantdoc_idx_to_pv_idx = {}
for pd_idx, pd_class_name in enumerate(plantdoc_dataset.classes):
    pv_class_name = PLANTDOC_TO_PLANTVILLAGE[pd_class_name]
    pv_idx = class_names.index(pv_class_name)
    plantdoc_idx_to_pv_idx[pd_idx] = pv_idx

print("\nIndex mapping built successfully.")

PlantDoc test set: 252 images, 27 classes
['Apple_Scab_Leaf', 'Apple_leaf', 'Apple_rust_leaf', 'Bell_pepper_leaf', 'Bell_pepper_leaf_spot', 'Blueberry_leaf', 'Cherry_leaf', 'Corn_Gray_leaf_spot', 'Corn_leaf_blight', 'Corn_rust_leaf', 'Peach_leaf', 'Potato_leaf_early_blight', 'Potato_leaf_late_blight', 'Raspberry_leaf', 'Soyabean_leaf', 'Squash_Powdery_mildew_leaf', 'Strawberry_leaf', 'Tomato_Early_blight_leaf', 'Tomato_Septoria_leaf_spot', 'Tomato_leaf', 'Tomato_leaf_bacterial_spot', 'Tomato_leaf_late_blight', 'Tomato_leaf_mosaic_virus', 'Tomato_leaf_yellow_virus', 'Tomato_mold_leaf', 'grape_leaf', 'grape_leaf_black_rot']

Index mapping built successfully.


In [3]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for images, labels in plantdoc_loader:
        images = images.to(device)

        # Remap PlantDoc ground-truth labels to PlantVillage label space
        remapped_labels = torch.tensor([plantdoc_idx_to_pv_idx[l.item()] for l in labels]).to(device)

        outputs = model(images)
        _, predicted = outputs.max(1)

        total += remapped_labels.size(0)
        correct += predicted.eq(remapped_labels).sum().item()

plantdoc_acc = 100. * correct / total
print(f"PlantDoc (field) accuracy: {plantdoc_acc:.2f}%")
print(f"PlantVillage (lab) accuracy: {checkpoint['val_acc']:.2f}%")
print(f"Lab-to-field gap: {checkpoint['val_acc'] - plantdoc_acc:.2f} percentage points")

PlantDoc (field) accuracy: 25.00%
PlantVillage (lab) accuracy: 93.87%
Lab-to-field gap: 68.87 percentage points
